# 05. Model Explainability with SHAP

This notebook:
1. Loads the trained model and preprocessor
2. Computes SHAP values for explainability
3. Generates per-student risk factor explanations
4. Creates visualizations for the dashboard
5. Saves explanation artifacts for production use

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Try to import shap
try:
    import shap
    HAS_SHAP = True
    print(f"SHAP version: {shap.__version__}")
except ImportError:
    HAS_SHAP = False
    print("SHAP not installed. Install with: pip install shap")

## 1. Load Model and Data

In [ ]:
# Load trained model and preprocessors
model = joblib.load('../models/classifier.joblib')
preprocessor = joblib.load('../models/preprocessor.joblib')
label_encoder = joblib.load('../models/label_encoder.joblib')
feature_config = joblib.load('../models/feature_config.joblib')

print(f"Model: {feature_config.get('model_name', 'Unknown')}")
print(f"Model Type: {feature_config.get('model_type', 'Unknown')}")
print(f"Features: {feature_config['all_features']}")
print(f"Classes: {list(label_encoder.classes_)}")

In [ ]:
# Load original data for explanations
df = pd.read_csv('../data/raw/synthetic_students.csv')

# Apply feature engineering
def create_features(df):
    df = df.copy()
    df['engagement_score'] = df['attendance_percentage'] * 0.6 + df['assignment_submission_rate'] * 0.4
    df['performance_index'] = df['internal_marks'] - df['engagement_score']
    df['low_attendance_flag'] = (df['attendance_percentage'] < 60).astype(int)
    df['low_marks_flag'] = (df['internal_marks'] < 50).astype(int)
    df['low_assignment_flag'] = (df['assignment_submission_rate'] < 50).astype(int)
    df['multiple_risk_flags'] = df['low_attendance_flag'] + df['low_marks_flag'] + df['low_assignment_flag']
    return df

df_engineered = create_features(df)

# Use a sample for SHAP analysis (faster)
sample_df = df_engineered.sample(n=100, random_state=42)
X_sample = sample_df[feature_config['all_features']]
X_sample_scaled = preprocessor.transform(X_sample)

print(f"\nSample size: {len(X_sample)} students")

## 2. SHAP Analysis

In [ ]:
if HAS_SHAP:
    # Create SHAP explainer
    explainer = shap.Explainer(model, X_sample_scaled)
    shap_values = explainer(X_sample_scaled)
    
    print("SHAP values computed successfully!")
    print(f"SHAP values shape: {shap_values.values.shape}")
    
    # Save explainer for production
    shap_data = {
        'explainer': explainer,
        'background_data': X_sample_scaled,
        'feature_names': feature_config['all_features']
    }
    joblib.dump(shap_data, '../models/shap_explainer.joblib')
    print("\nSaved SHAP explainer: ../models/shap_explainer.joblib")

## 3. Global Feature Importance

In [ ]:
if HAS_SHAP:
    # Summary plot (bar)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample,
                      feature_names=feature_config['all_features'],
                      plot_type="bar", show=False)
    plt.title('Global Feature Importance (SHAP)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../models/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if HAS_SHAP:
    # Beeswarm plot
    plt.figure(figsize=(12, 6))
    shap.summary_plot(shap_values, X_sample,
                      feature_names=feature_config['all_features'],
                      show=False)
    plt.title('SHAP Summary Plot', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../models/shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Per-Student Risk Factor Explanation

This function can be used in production to explain individual predictions.

In [ ]:
def get_student_explanation(student_data, student_name=None):
    """
    Generate explanation for a single student's risk prediction.
    
    Parameters:
    -----------
    student_data : dict or pd.Series
        Student features (raw values before preprocessing)
    student_name : str, optional
        Student name for display
    
    Returns:
    --------
    dict
        Contains risk_level, probabilities, and top_risk_factors
    """
    # Prepare input
    X_student = pd.DataFrame([student_data])[feature_config['all_features']]
    X_student_scaled = preprocessor.transform(X_student)
    
    # Predict
    pred_idx = model.predict(X_student_scaled)[0]
    pred_proba = model.predict_proba(X_student_scaled)[0]
    
    risk_level = label_encoder.inverse_transform([pred_idx])[0]
    
    # Get SHAP values for this student
    if HAS_SHAP:
        # Compute SHAP for this single instance
        explainer = shap.Explainer(model, X_sample_scaled)
        shap_vals = explainer(X_student_scaled)
        
        # Get absolute SHAP values for ranking
        mean_shap = np.abs(shap_vals.values).mean(axis=2)[0]
        
        # Rank features by importance
        feature_importance = list(zip(
            feature_config['all_features'],
            mean_shap,
            shap_vals.values[0, :, pred_idx]
        ))
        feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)
        
        # Build explanation
        top_factors = []
        for feat, importance, direction in feature_importance[:3]:
            value = student_data[feat]
            if direction > 0:
                impact = "increases risk"
            else:
                impact = "decreases risk"
            top_factors.append({
                'factor': feat,
                'value': round(value, 2),
                'impact': impact,
                'direction': round(direction, 4)
            })
    else:
        # Fallback: rule-based explanation
        top_factors = []
        if student_data.get('attendance_percentage', 100) < 60:
            top_factors.append({'factor': 'attendance_percentage',
                               'value': student_data['attendance_percentage'],
                               'impact': 'increases risk', 'direction': 1})
        if student_data.get('internal_marks', 100) < 50:
            top_factors.append({'factor': 'internal_marks',
                               'value': student_data['internal_marks'],
                               'impact': 'increases risk', 'direction': 1})
        if student_data.get('assignment_submission_rate', 100) < 50:
            top_factors.append({'factor': 'assignment_submission_rate',
                               'value': student_data['assignment_submission_rate'],
                               'impact': 'increases risk', 'direction': 1})
    
    return {
        'student_name': student_name,
        'student_id': student_data.get('student_id', 'Unknown'),
        'risk_level': risk_level,
        'probabilities': dict(zip(label_encoder.classes_, pred_proba.round(3))),
        'top_risk_factors': top_factors
    }

# Test with a few students
for i in [0, 1, 2]:
    student = df_engineered.iloc[i]
    explanation = get_student_explanation(
        student[feature_config['all_features']],
        student['student_name']
    )
    print(f"\n{'='*50}")
    print(f"Student: {explanation['student_name']} ({explanation['student_id']})")
    print(f"Risk Level: {explanation['risk_level']}")
    print(f"Probabilities: {explanation['probabilities']}")
    print("Top Risk Factors:")
    for factor in explanation['top_risk_factors']:
        print(f"  - {factor['factor']}: {factor['value']} ({factor['impact']})")

## 5. Generate Explanations for All Students

In [ ]:
# Generate explanations for all students
all_explanations = []

for idx, student in df_engineered.iterrows():
    explanation = get_student_explanation(
        student[feature_config['all_features']],
        student['student_name']
    )
    all_explanations.append(explanation)

# Create explanations DataFrame
explanations_df = pd.DataFrame([
    {
        'student_id': df_engineered.iloc[i]['student_id'],
        'student_name': df_engineered.iloc[i]['student_name'],
        'risk_level': exp['risk_level'],
        'prob_low': exp['probabilities'].get('Low', 0),
        'prob_moderate': exp['probabilities'].get('Moderate', 0),
        'prob_high': exp['probabilities'].get('High', 0),
        'top_factor': exp['top_risk_factors'][0]['factor'] if exp['top_risk_factors'] else None,
        'top_factor_value': exp['top_risk_factors'][0]['value'] if exp['top_risk_factors'] else None,
        'top_factor_impact': exp['top_risk_factors'][0]['impact'] if exp['top_risk_factors'] else None
    }
    for i, exp in enumerate(all_explanations)
])

explanations_df.to_csv('../data/processed/student_explanations.csv', index=False)
print(f"Saved explanations for {len(explanations_df)} students")
explanations_df.head()

## 6. Example: Explain a High-Risk Student

In [ ]:
# Find a high-risk student and explain
high_risk_student = df_engineered[df_engineered['risk_level'] == 'High'].iloc[0]

print("\n" + "="*60)
print("HIGH-RISK STUDENT EXPLANATION EXAMPLE")
print("="*60)

explanation = get_student_explanation(
    high_risk_student[feature_config['all_features']],
    high_risk_student['student_name']
)

print(f"\nStudent: {explanation['student_name']}")
print(f"ID: {explanation['student_id']}")
print(f"Predicted Risk: {explanation['risk_level']}")
print(f"\nProbability Distribution:")
for level, prob in explanation['probabilities'].items():
    bar = '█' * int(prob * 20)
    print(f"  {level}: {prob:.1%} {bar}")

print(f"\nTop Risk Factors:")
for i, factor in enumerate(explanation['top_risk_factors'], 1):
    print(f"  {i}. {factor['factor']}: {factor['value']} ({factor['impact']})")

## 7. Summary of Saved Artifacts

In [ ]:
import os

print("\n" + "="*60)
print("EXPLAINABILITY ANALYSIS COMPLETE")
print("="*60)

print("\nFiles in ../models/:")
for f in sorted(os.listdir('../models/')):
    size = os.path.getsize(f'../models/{f}')
    print(f"  {f}: {size:,} bytes")

print("\nFiles in ../data/processed/:")
for f in sorted(os.listdir('../data/processed/')):
    size = os.path.getsize(f'../data/processed/{f}')
    print(f"  {f}: {size:,} bytes")

print("\n" + "="*60)
print("PRODUCTION USAGE")
print("="*60)
print("""
To explain predictions in production:

    import joblib
    
    model = joblib.load('models/classifier.joblib')
    preprocessor = joblib.load('models/preprocessor.joblib')
    label_encoder = joblib.load('models/label_encoder.joblib')
    
    # For SHAP explanations:
    shap_data = joblib.load('models/shap_explainer.joblib')
    explainer = shap_data['explainer']
""")